# FedRAG — Build & Query FAISS index (SQ8 + BQ/Rescore) trên Colab

Notebook này build FAISS index cho 4 corpus của FedRAG (StatPearls, Textbooks, PubMed, Wikipedia) và lưu thẳng vào Google Drive:
- **SQ8** (`IndexIVFScalarQuantizer`) cho StatPearls & Textbooks (tập nhỏ).
- **BQ + Rescore 3x** (`IndexBinaryFlat` + memmap float32) cho PubMed & Wikipedia (tập lớn, 54.2 triệu vector).

Bảng dự toán dung lượng chi tiết (~175 GB tổng, cần Google One 2TB): xem artifact đã gửi trong chat.

**Chạy các cell theo đúng thứ tự.** Trước khi chạy notebook này, đảm bảo bạn đã `git push` bản refactor `retriever.py` lên GitHub.

## 0. Cấu hình repo
Thay `REPO_URL` nếu bạn fork/đổi remote khác.

In [ ]:
REPO_URL = "https://github.com/ursuswh-metamorphic/Rag_Router_Reproduce.git"
BRANCH = "main"


## 1. Mount Google Drive an toàn
Dùng 1 thư mục con riêng, kiểm tra dung lượng trống trước khi build PubMed/Wikipedia (~175 GB).

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, shutil

DRIVE_ROOT = "/content/drive/MyDrive/fedrag_corpus"
os.makedirs(DRIVE_ROOT, exist_ok=True)

total, used, free = shutil.disk_usage(DRIVE_ROOT)
print(f"Google Drive free space: {free / 1e9:.1f} GB")
# PubMed + Wikipedia can ~175 GB - dung lai neu free < 200 GB


## 2. Cài đặt môi trường (git-lfs, clone repo, dependencies)
PubMed/Wikipedia được clone qua **git-lfs** từ HuggingFace. Dự án dùng `faiss-cpu` (không cần `faiss-gpu`: GPU chỉ dùng để encode MedCPT qua torch).

In [ ]:
!apt-get -qq update && apt-get -qq install -y git-lfs
!git lfs install

!git clone --branch "$BRANCH" "$REPO_URL" fedrag
%cd fedrag
!pip install -q -r requirements.txt


## 3. Cấu hình đường dẫn & tham số môi trường
`FEDRAG_CORPUS_DIR` trỏ vào Drive; các biến `FEDRAG_*` khác điều chỉnh shard size / oversample / AMP.

In [ ]:
import sys, os
sys.path.insert(0, "/content/fedrag")

os.environ["FEDRAG_CORPUS_DIR"] = DRIVE_ROOT
os.environ["FEDRAG_FAISS_SHARD_FILES"] = "10"      # nho hon default (25) -> ha RAM dinh khi flush shard
os.environ["FEDRAG_BQ_OVERSAMPLE_FACTOR"] = "3"    # tang rescore BQ lay knn*3 ung vien
os.environ["FEDRAG_ENABLE_AMP"] = "1"               # fp16 khi encode MedCPT tren T4/A100


## 4a. Build nhanh: StatPearls & Textbooks (SQ8, chạy trực tiếp trong notebook)
Dùng `data/prepare.sh` — tự động download corpus + build index + chạy 1 query mẫu.

In [ ]:
!bash data/prepare.sh \
  --datasets statpearls textbooks \
  --index_num_chunks 0 \
  --storage_dir "$DRIVE_ROOT" \
  --batch_size 128


## 4b. (Tuỳ chọn) Gọi trực tiếp `build_faiss_index` / `query_faiss_index`
Dùng khi corpus đã được download sẵn và bạn chỉ muốn build lại index hoặc test query.

In [ ]:
from fedrag.retriever import Retriever

retriever = Retriever(corpus_dir=DRIVE_ROOT)
retriever.build_faiss_index(dataset_name="textbooks", batch_size=64, num_chunks=None)

result = retriever.query_faiss_index(
    dataset_name="textbooks",
    query="What are the complications of a cardiovascular disease?",
    knn=5,
)
for doc_id, info in result.items():
    print(f"#{info['rank']}  score={info['score']:.4f}  {info['title']}")


## 5. Build PubMed (BQ + Rescore) — chạy nền, chống Colab disconnect
**Cảnh báo:** `build_faiss_index()` xoá sạch shard cũ trước khi build lại (không resume tự động). Corpus 23.9M vector, cần Runtime **High-RAM** (Runtime → Change runtime type). Chạy nền bằng `nohup` để job sống sót qua việc mất kết nối *frontend* (đóng tab/rớt mạng trình duyệt) — không sống sót qua việc Colab thu hồi hẳn VM runtime.

In [ ]:
os.environ["FEDRAG_FAISS_SHARD_FILES"] = "8"  # ha them cho corpus 23.9M vector

pubmed_log = "/content/pubmed_build.log"
!nohup python -m data.prepare \
    --datasets pubmed --index_num_chunks 0 \
    --storage_dir "$DRIVE_ROOT" --download_workers 2 --batch_size 96 \
    > {pubmed_log} 2>&1 &
print("Dang build PubMed o nen, theo doi log:", pubmed_log)


### Theo dõi tiến độ PubMed
Chạy lại cell này bất cứ lúc nào.

In [ ]:
!tail -n 30 /content/pubmed_build.log
!free -h
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


## 6. Build Wikipedia (BQ + Rescore) — chạy SAU khi PubMed xong
Không chạy song song với PubMed (cả hai đều ăn RAM mạnh lúc flush shard). Kiểm tra `pubmed_build.log` báo hoàn tất trước khi chạy cell dưới.

In [ ]:
wiki_log = "/content/wikipedia_build.log"
!nohup python -m data.prepare \
    --datasets wikipedia --index_num_chunks 0 \
    --storage_dir "$DRIVE_ROOT" --download_workers 2 --batch_size 96 \
    > {wiki_log} 2>&1 &
print("Dang build Wikipedia o nen, theo doi log:", wiki_log)


In [ ]:
!tail -n 30 /content/wikipedia_build.log
!free -h


## 7. Query thử sau khi build xong


In [ ]:
from fedrag.retriever import Retriever

retriever = Retriever(corpus_dir=DRIVE_ROOT)
for name in ("statpearls", "textbooks", "pubmed", "wikipedia"):
    try:
        res = retriever.query_faiss_index(name, "What are the complications of a cardiovascular disease?", knn=3)
        print(name, "->", list(res.keys()))
    except Exception as e:
        print(name, "chua san sang:", e)
